# Ноутбук 02: Генерация шума и загрязнение базы знаний

**Цель:** подготовить все артефакты шума, которые будут использоваться в ноутбуках 03+ при двух режимах эксперимента (controlled-context и end-to-end).

## Что делает этот ноутбук

1. Подключает артефакты из ноутбука 01 (questions, gold_mapping, distractor_mapping).
2. Генерирует **4 типа шума** для каждого из 100 вопросов:
   - **Semantic distractors** — берутся готовыми из HotpotQA (без LLM).
   - **Counterfactuals** — через Groq LLM создаём «правдоподобно ложные» версии gold-параграфов.
   - **Duplicates** — через Groq LLM создаём перефразировки gold-параграфов.
   - **Structural** — детерминированное искажение (HTML-мусор, OCR-ошибки, обрывы) — без LLM.
3. Проводит **валидацию** сгенерированного шума (факт реально изменён, похоже на оригинал, не выродилось в бессмыслицу).
4. Сохраняет всё в `noise_cache.json` на Google Drive.
5. Создаёт в Qdrant **вторую коллекцию `rag_noisy`**: копия `rag_clean` + инъекции контрафактов/дубликатов/структурных искажений. Используется в end-to-end режиме.
6. Проводит sanity-check: находит ли retriever инъецированные шумные документы в реальном пайплайне.

## Ключевые параметры

- **Уровни шума** в эксперименте: 0%, 40%, 80% (закрепится в ноутбуках 03+).
- **Модель для генерации:** `llama-3.3-70b-versatile` через Groq.
- **Лимиты Groq free tier:** ~6000 TPM, 500k tokens/day. Поэтому в ноутбуке встроены rate-limiter и retry-логика.
- **Контрафакты на вопрос:** 2 (по одному на каждый gold-параграф).
- **Дубликаты на вопрос:** 2 (аналогично).
- **Структурные искажения на вопрос:** 1 (искажается один случайный gold).

## Шаг 0. Установка и инициализация

In [1]:
!pip install -q qdrant-client fastembed groq tqdm tenacity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 33.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import time
import random
import hashlib
import re
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import numpy as np
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# ---- КОНФИГ ----
# ВАЖНО: используйте НОВЫЕ ключи (старые скомпрометированы)
QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")

assert QDRANT_URL, "Не найден QDRANT_URL"
assert QDRANT_API_KEY, "Не найден QDRANT_API_KEY"
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, "Не найден GROQ_API_KEY"

SEED = 42

# Модель для генерации шума
GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_TEMPERATURE = 0.3  # немного вариативности для разнообразия контрафактов

# Rate limiting (Groq free tier: ~6000 TPM для 70B)
# Консервативно ограничимся 5500 TPM чтобы был запас
GROQ_TPM_BUDGET = 5500
GROQ_REQUEST_DELAY = 0.5  # секунд между запросами как базовая задержка

# Параметры генерации шума
N_COUNTERFACTUALS_PER_GOLD = 1  # 1 контрафакт на каждый gold-параграф (итого 2 на вопрос)
N_DUPLICATES_PER_GOLD = 1       # аналогично
MAX_GENERATION_ATTEMPTS = 3     # макс. попыток регенерации если валидация не прошла

# Коллекции в Qdrant
COLLECTION_CLEAN = "rag_clean"
COLLECTION_NOISY = "rag_noisy"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM = 384
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

# Пути к артефактам
ARTIFACTS_DIR = Path("/content/drive/MyDrive/rag_experiment/artifacts")
assert ARTIFACTS_DIR.exists(), f"Не найдена папка с артефактами: {ARTIFACTS_DIR}"

NOISE_CACHE_PATH = ARTIFACTS_DIR / "noise_cache.json"
NOISE_VALIDATION_REPORT_PATH = ARTIFACTS_DIR / "noise_validation_report.json"

random.seed(SEED)
np.random.seed(SEED)

print(f"Artifacts: {ARTIFACTS_DIR}")
print(f"Groq model: {GROQ_MODEL}")
print(f"Seed: {SEED}")

Artifacts: /content/drive/MyDrive/rag_experiment/artifacts
Groq model: llama-3.3-70b-versatile
Seed: 42


In [4]:
# Загружаем артефакты из ноутбука 01
with open(ARTIFACTS_DIR / "questions.json", encoding="utf-8") as f:
    questions = json.load(f)
with open(ARTIFACTS_DIR / "gold_mapping.json", encoding="utf-8") as f:
    gold_mapping = json.load(f)
with open(ARTIFACTS_DIR / "distractor_mapping.json", encoding="utf-8") as f:
    distractor_mapping = json.load(f)
with open(ARTIFACTS_DIR / "config.json", encoding="utf-8") as f:
    base_config = json.load(f)

print(f"Вопросов загружено: {len(questions)}")
print(f"Gold mappings: {len(gold_mapping)}")
print(f"Distractor mappings: {len(distractor_mapping)}")
print(f"Базовый конфиг: {base_config}")

Вопросов загружено: 100
Gold mappings: 100
Distractor mappings: 100
Базовый конфиг: {'seed': 42, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'embedding_dim': 384, 'chunk_size': 512, 'chunk_overlap': 64, 'collection_clean': 'rag_clean', 'qdrant_url': 'https://92fb2201-8122-4f15-aa86-326522e1fcea.eu-central-1-0.aws.cloud.qdrant.io', 'dataset': 'hotpotqa/hotpot_qa:distractor:validation', 'n_questions': 100, 'n_background': 400}


## Шаг 1. Подключение к сервисам

In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType, Filter, FieldCondition, MatchValue
from groq import Groq, RateLimitError, APIError
from fastembed import TextEmbedding

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
groq_client = Groq(api_key=GROQ_API_KEY)

print("Инициализация embedder...")
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

# Проверка коллекции rag_clean из ноутбука 01
clean_info = qdrant.get_collection(COLLECTION_CLEAN)
print(f"\nrag_clean: {clean_info.points_count} точек, статус {clean_info.status}")
assert clean_info.points_count > 0, "rag_clean пустая — запустите сначала ноутбук 01"

Инициализация embedder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


rag_clean: 7997 точек, статус green


## Шаг 2. Rate-limited обёртка для Groq

Free tier даёт ~6000 TPM на 70B-модели. Если превысить — 429 Too Many Requests. Поэтому:

- **Rate limiter** следит за расходом токенов в скользящем окне 60 секунд.
- **Retry с экспоненциальным бэкофом** через `tenacity` для случаев, когда Groq всё-таки вернёт 429 (например, при неточной оценке токенов).
- Между запросами базовая задержка 0.5с.

In [6]:
class GroqTokenBudget:
    """Скользящее окно для учёта расхода токенов. Блокирует, если скоро превысим лимит."""
    def __init__(self, tpm_limit: int):
        self.tpm_limit = tpm_limit
        self.usage = []  # список (timestamp, tokens)

    def _prune(self):
        cutoff = time.time() - 60
        self.usage = [(t, n) for t, n in self.usage if t > cutoff]

    def wait_if_needed(self, estimated_tokens: int):
        self._prune()
        used = sum(n for _, n in self.usage)
        if used + estimated_tokens > self.tpm_limit:
            # ждём до момента, когда самая старая запись «выйдет» из окна
            if self.usage:
                oldest_ts = self.usage[0][0]
                wait_sec = max(0, 60 - (time.time() - oldest_ts)) + 1
                print(f"  [rate-limiter] used={used}, est={estimated_tokens}, жду {wait_sec:.1f}с...")
                time.sleep(wait_sec)
                self._prune()

    def record(self, tokens: int):
        self.usage.append((time.time(), tokens))

budget = GroqTokenBudget(tpm_limit=GROQ_TPM_BUDGET)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    retry=retry_if_exception_type((RateLimitError, APIError)),
    reraise=True,
)
def groq_chat(messages, response_format=None, max_tokens=800, temperature=GROQ_TEMPERATURE):
    """Обёртка над Groq API с учётом rate limit."""
    # Грубая оценка входных токенов: 1 токен ~ 4 символа
    input_chars = sum(len(m["content"]) for m in messages)
    est_tokens = input_chars // 4 + max_tokens
    budget.wait_if_needed(est_tokens)

    kwargs = dict(
        model=GROQ_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    if response_format is not None:
        kwargs["response_format"] = response_format

    resp = groq_client.chat.completions.create(**kwargs)
    # Записываем фактический расход
    actual_tokens = resp.usage.total_tokens if resp.usage else est_tokens
    budget.record(actual_tokens)

    time.sleep(GROQ_REQUEST_DELAY)
    return resp.choices[0].message.content, actual_tokens

# Быстрый тест связи
text, tokens = groq_chat([{"role": "user", "content": "Say hi in 3 words"}], max_tokens=50)
print(f"Тест Groq OK: '{text.strip()}' ({tokens} tokens)")

Тест Groq OK: 'Hello to you' (45 tokens)


## Шаг 3. Генератор 1 — семантические дистракторы

**Без LLM.** Берём готовые distractor-параграфы из HotpotQA (уже сохранены в `distractor_mapping.json`). Каждый вопрос имеет 8 distractor'ов — это связанные по теме параграфы из соседних статей Википедии, которые НЕ содержат ответа.

Почему это хороший шум:
- Они **семантически близки** к запросу (высокий cosine к вопросу), что делает их идеальными «обманщиками».
- Они **реальные**, не синтетические — на них тренируются все retrieval-системы.
- Они прошли разметку авторов HotpotQA как «distractors» — то есть их ложность подтверждена людьми.

In [7]:
def get_semantic_distractors(question_id, n=4):
    """Возвращает до n дистракторов из HotpotQA для данного вопроса."""
    distractors = distractor_mapping.get(question_id, {})
    items = list(distractors.items())
    # Детерминированный отбор для воспроизводимости
    rng = random.Random(SEED + hash(question_id) % 10000)
    rng.shuffle(items)
    return [
        {"title": title, "text": text, "source": "hotpot_distractor"}
        for title, text in items[:n]
    ]

# Пробуем на первом вопросе
q0 = questions[0]
distractors_q0 = get_semantic_distractors(q0["id"], n=3)
print(f"Вопрос: {q0['question']}")
print(f"Получено distractor'ов: {len(distractors_q0)}")
for d in distractors_q0:
    print(f"  - {d['title']}: {d['text'][:100]}...")

Вопрос: Where was the Super Bowl, that Alan Faneca won, played ?
Получено distractor'ов: 3
  - 2007 New York Giants season: The 2007 New York Giants season was the 83rd season for the New York Giants in the National Football...
  - Super Bowl XXVII: Super Bowl XXVII was an American football game between the American Football Conference (AFC) champi...
  - 2009 New York Jets season: The 2009 New York Jets season was the 50th season for the club and the 40th season in the National F...


## Шаг 4. Генератор 2 — контрафакты (через Groq)

**Что такое контрафакт:** переписанный gold-параграф, в котором ключевой факт (имя, дата, число, место) заменён на правдоподобный ложный. Остальной текст максимально сохранён.

**Зачем:** это самый опасный тип шума. Retriever не может отличить контрафакт от правды (он семантически такой же), а LLM может поверить контрафакту и дать неверный ответ.

**Структурированный вывод:** используем `response_format={"type": "json_object"}` для Groq, чтобы получать JSON со всеми нужными полями — это важно для валидации.

In [16]:
def build_counterfactual_prompt(question, answer, paragraph):
    return f"""You are a test-data generator for a research project on noise-robust RAG systems.

Your task: rewrite the paragraph below so that it incorrectly answers the given question, while remaining plausible and well-written.

Rules:
1. Change EXACTLY ONE key fact (a name, date, number, or location) that is relevant to answering the question.
2. Replace it with a plausible but incorrect value (e.g., change a real year to another real-sounding year; change a city to another real city).
3. Keep the rest of the paragraph as close to the original as possible — same style, same structure, same length.
4. Do NOT use absurd replacements (e.g., do not put things on Mars, do not use year 3000).
5. Do NOT flag the change or add disclaimers — the output paragraph must read as a normal encyclopedic text.

Return a JSON object with exactly these fields:
{{
  "original_value": "<the original fact, verbatim from the paragraph>",
  "new_value": "<the plausible replacement you chose>",
  "reasoning": "<one short sentence explaining why this fact is answer-relevant>",
  "new_text": "<the full rewritten paragraph with the change applied>"
}}

Question: {question}

Correct answer (for your reference): {answer}

Paragraph to modify:
{paragraph}
"""

from groq import BadRequestError

def generate_counterfactual(question, answer, paragraph_text):
    """Одна попытка генерации контрафакта. Возвращает (dict_or_None, tokens)."""
    prompt = build_counterfactual_prompt(question, answer, paragraph_text)
    try:
        text, tokens = groq_chat(
            [{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            max_tokens=900,
        )
        data = json.loads(text)
        required = {"original_value", "new_value", "reasoning", "new_text"}
        if not required.issubset(data.keys()):
            return None, tokens
        return data, tokens
    except (json.JSONDecodeError, KeyError) as e:
        print(f"    Parse error: {e}")
        return None, 0
    except BadRequestError as e:
        # Groq отказался отдавать ответ из-за невалидного JSON в выходе модели.
        # Это происходит, когда модель вставляет литеральные кавычки/спецсимволы в строки.
        # Считаем как failed attempt и идём на ретрай.
        print(f"    Groq JSON validation failed (will retry): {str(e)[:120]}")
        return None, 0

In [9]:
def validate_counterfactual(cf_data, original_text):
    """Проверяет качество контрафакта.

    Возвращает (is_valid, reason).
    """
    orig_val = cf_data["original_value"].strip()
    new_val = cf_data["new_value"].strip()
    new_text = cf_data["new_text"].strip()

    # 1. Значения должны реально отличаться
    if orig_val.lower() == new_val.lower():
        return False, "original_value == new_value"

    # 2. Новый текст должен быть непустым и разумной длины
    if len(new_text) < 50:
        return False, "new_text too short"
    if len(new_text) > len(original_text) * 2:
        return False, "new_text too long (likely hallucinated extra content)"

    # 3. Новое значение должно присутствовать в новом тексте (иначе замена формальная)
    if new_val.lower() not in new_text.lower():
        return False, "new_value not in new_text"

    # 4. Семантическое сходство с оригиналом (должно быть высоким, иначе переписал слишком сильно)
    vec_orig = list(embedder.embed([original_text]))[0]
    vec_new = list(embedder.embed([new_text]))[0]
    cos_sim = np.dot(vec_orig, vec_new) / (np.linalg.norm(vec_orig) * np.linalg.norm(vec_new))
    if cos_sim < 0.75:
        return False, f"low cosine similarity ({cos_sim:.3f})"

    return True, f"ok (cos={cos_sim:.3f})"

def generate_validated_counterfactual(question, answer, paragraph_text, max_attempts=MAX_GENERATION_ATTEMPTS):
    """Генерирует контрафакт с валидацией, до max_attempts попыток."""
    history = []
    for attempt in range(1, max_attempts + 1):
        data, tokens = generate_counterfactual(question, answer, paragraph_text)
        if data is None:
            history.append({"attempt": attempt, "status": "parse_failed"})
            continue
        is_valid, reason = validate_counterfactual(data, paragraph_text)
        history.append({"attempt": attempt, "status": "valid" if is_valid else "invalid", "reason": reason})
        if is_valid:
            data["validation_passed"] = True
            data["attempts"] = attempt
            data["history"] = history
            return data
    # Все попытки провалились — возвращаем последнюю с пометкой
    if data is not None:
        data["validation_passed"] = False
        data["attempts"] = max_attempts
        data["history"] = history
    return data  # может быть None если ни разу не распарсилось

In [10]:
# Пробный запуск на одном вопросе
q_test = questions[0]
gold = gold_mapping[q_test["id"]]
first_gold_title = gold["gold_titles"][0]
first_gold_text = gold["gold_texts"][first_gold_title]

print(f"Тест на вопросе: {q_test['question']}")
print(f"Ответ: {q_test['answer']}")
print(f"Gold title: {first_gold_title}")
print(f"Gold text: {first_gold_text[:200]}...\n")

cf = generate_validated_counterfactual(q_test["question"], q_test["answer"], first_gold_text)
if cf:
    print(f"original_value: {cf['original_value']}")
    print(f"new_value: {cf['new_value']}")
    print(f"reasoning: {cf['reasoning']}")
    print(f"validation_passed: {cf['validation_passed']}")
    print(f"attempts: {cf['attempts']}")
    print(f"new_text: {cf['new_text'][:300]}...")
else:
    print("Не удалось сгенерировать")

Тест на вопросе: Where was the Super Bowl, that Alan Faneca won, played ?
Ответ: Ford Field in Detroit
Gold title: Alan Faneca
Gold text: Alan Joseph Faneca ( ; born December 7, 1976) is a former professional American football player who was a guard in the National Football League (NFL) for thirteen seasons.  He played college football ...

original_value: Ford Field in Detroit
new_value: Lambeau Field in Green Bay
reasoning: The location of the Super Bowl is relevant to answering where Alan Faneca won the championship.
validation_passed: True
attempts: 1
new_text: Alan Joseph Faneca ( ; born December 7, 1976) is a former professional American football player who was a guard in the National Football League (NFL) for thirteen seasons.  He played college football for Louisiana State University (LSU), and earned consensus All-America honors.  He was drafted by th...


In [11]:
# Пробный запуск на одном вопросе
q_test = questions[1]
gold = gold_mapping[q_test["id"]]
first_gold_title = gold["gold_titles"][1]
first_gold_text = gold["gold_texts"][first_gold_title]

print(f"Тест на вопросе: {q_test['question']}")
print(f"Ответ: {q_test['answer']}")
print(f"Gold title: {first_gold_title}")
print(f"Gold text: {first_gold_text[:200]}...\n")

cf = generate_validated_counterfactual(q_test["question"], q_test["answer"], first_gold_text)
if cf:
    print(f"original_value: {cf['original_value']}")
    print(f"new_value: {cf['new_value']}")
    print(f"reasoning: {cf['reasoning']}")
    print(f"validation_passed: {cf['validation_passed']}")
    print(f"attempts: {cf['attempts']}")
    print(f"new_text: {cf['new_text'][:300]}...")
else:
    print("Не удалось сгенерировать")

Тест на вопросе: National Firearms Agreement was in response to the Port Arthur massacre that killed how many people?
Ответ: 35 people
Gold title: Port Arthur massacre (Australia)
Gold text: The Port Arthur massacre of 28–29 April 1996 was a massacre in which 35 people were killed and 23 wounded.  It occurred mainly at the historic Port Arthur former prison colony, a popular tourist site ...

original_value: 35
new_value: 32
reasoning: The number of people killed is directly relevant to understanding the severity of the Port Arthur massacre and its impact on the National Firearms Agreement
validation_passed: True
attempts: 1
new_text: The Port Arthur massacre of 28–29 April 1996 was a massacre in which 32 people were killed and 23 wounded.  It occurred mainly at the historic Port Arthur former prison colony, a popular tourist site in south-eastern Tasmania, Australia.  It was the deadliest mass shooting in Australian history, and...


## Шаг 5. Генератор 3 — дубликаты (через Groq)

**Что такое дубликат:** перефразированная версия gold-параграфа. Все факты сохранены, но другими словами.

**Зачем:** проверить эффект «заполнения контекста повторами» — когда retriever возвращает несколько семантически похожих копий одного факта, вытесняя другие важные документы. Это шум типа *redundancy*.

In [17]:
def build_duplicate_prompt(paragraph):
    return f"""You are rewriting an encyclopedia paragraph in different words while preserving ALL factual content.

Rules:
1. Keep every fact — names, dates, numbers, places, relationships — EXACTLY the same.
2. Change the wording: use synonyms, restructure sentences, reorder clauses.
3. Do NOT add new information that is not in the original.
4. Do NOT remove information.
5. Keep roughly the same length.

Return a JSON object:
{{
  "new_text": "<the paraphrased paragraph>"
}}

Original paragraph:
{paragraph}
"""

def generate_duplicate(paragraph_text):
    prompt = build_duplicate_prompt(paragraph_text)
    try:
        text, tokens = groq_chat(
            [{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            max_tokens=800,
        )
        data = json.loads(text)
        if "new_text" not in data:
            return None
        return data
    except (json.JSONDecodeError, KeyError):
        return None
    except BadRequestError as e:
        print(f"    Groq JSON validation failed for duplicate (will retry): {str(e)[:120]}")
        return None
def validate_duplicate(dup_data, original_text):
    new_text = dup_data["new_text"].strip()

    # Не должно быть почти идентичным
    if new_text == original_text:
        return False, "identical to original"

    if len(new_text) < 50:
        return False, "new_text too short"
    if len(new_text) > len(original_text) * 2:
        return False, "new_text too long"

    # Cosine similarity должна быть высокой (> 0.85) — те же факты, разные слова
    vec_orig = list(embedder.embed([original_text]))[0]
    vec_new = list(embedder.embed([new_text]))[0]
    cos_sim = np.dot(vec_orig, vec_new) / (np.linalg.norm(vec_orig) * np.linalg.norm(vec_new))
    if cos_sim < 0.85:
        return False, f"low cosine similarity ({cos_sim:.3f}) — too divergent"
    if cos_sim > 0.995:
        return False, f"cosine too high ({cos_sim:.3f}) — almost identical"

    return True, f"ok (cos={cos_sim:.3f})"

def generate_validated_duplicate(paragraph_text, max_attempts=MAX_GENERATION_ATTEMPTS):
    history = []
    for attempt in range(1, max_attempts + 1):
        data = generate_duplicate(paragraph_text)
        if data is None:
            history.append({"attempt": attempt, "status": "parse_failed"})
            continue
        is_valid, reason = validate_duplicate(data, paragraph_text)
        history.append({"attempt": attempt, "status": "valid" if is_valid else "invalid", "reason": reason})
        if is_valid:
            data["validation_passed"] = True
            data["attempts"] = attempt
            data["history"] = history
            return data
    if data is not None:
        data["validation_passed"] = False
        data["attempts"] = max_attempts
        data["history"] = history
    return data

In [13]:
# Тест
dup = generate_validated_duplicate(first_gold_text)
if dup:
    print(f"validation_passed: {dup['validation_passed']}")
    print(f"attempts: {dup['attempts']}")
    print(f"new_text: {dup['new_text'][:300]}...")

validation_passed: True
attempts: 1
new_text: On 28–29 April 1996, a devastating massacre took place, resulting in the deaths of 35 individuals and injuring 23 others. This tragic event primarily occurred at the historic former prison colony of Port Arthur, a renowned tourist destination situated in south-eastern Tasmania, Australia. The incide...


## Шаг 6. Генератор 4 — структурный шум (без LLM)

Детерминированное искажение gold-параграфа. Имитирует реальные проблемы парсинга документов:

- **HTML-мусор** — вставка тегов, навигационных блоков, рекламы (как после парсинга веб-страниц).
- **OCR-ошибки** — замена похожих символов (как после распознавания скана).
- **Обрывы** — потеря куска текста (как при плохом chunking).

Все три техники применяются одновременно с небольшой интенсивностью, чтобы параграф остался узнаваемым, но «замусоренным».

In [14]:
HTML_JUNK_SNIPPETS = [
    '<div class="nav-bar">Home | About | Contact | Login</div>',
    '[Skip to main content] [Search] [Menu]',
    '<script>window.adTracker.init();</script>',
    '<!-- Ad slot 4712 -->',
    'Cookie Policy: We use cookies to improve your experience. [Accept] [Reject]',
    '[Edit] [View history] [Talk]',
    '</div></div><footer class="site-footer">',
    'Share: Facebook | Twitter | Reddit | Email',
]

OCR_SUBSTITUTIONS = [
    ("o", "0"), ("O", "0"), ("l", "1"), ("I", "l"),
    ("rn", "m"), ("m", "rn"), ("cl", "d"), ("vv", "w"),
    ("e", "c"), ("a", "o"),
]

def apply_ocr_noise(text, error_rate=0.02, rng=None):
    """Случайно заменяет символы на похожие с заданной вероятностью."""
    rng = rng or random
    result = []
    i = 0
    while i < len(text):
        if rng.random() < error_rate:
            # Пытаемся применить какую-нибудь подстановку
            applied = False
            for old, new in OCR_SUBSTITUTIONS:
                if text[i:i+len(old)] == old:
                    result.append(new)
                    i += len(old)
                    applied = True
                    break
            if not applied:
                result.append(text[i])
                i += 1
        else:
            result.append(text[i])
            i += 1
    return "".join(result)

def inject_html_junk(text, n_injections=2, rng=None):
    """Вставляет HTML/навигационный мусор в случайные места."""
    rng = rng or random
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) < 2:
        return text
    for _ in range(n_injections):
        pos = rng.randint(1, len(sentences) - 1)
        junk = rng.choice(HTML_JUNK_SNIPPETS)
        sentences.insert(pos, junk)
    return " ".join(sentences)

def truncate_sentences(text, rng=None, prob=0.15):
    """С вероятностью prob обрывает предложение на середине, ставя [...]."""
    rng = rng or random
    sentences = re.split(r'(?<=[.!?])\s+', text)
    result = []
    for s in sentences:
        if rng.random() < prob and len(s) > 30:
            cut = rng.randint(len(s) // 3, int(len(s) * 0.7))
            result.append(s[:cut] + " [...]")
        else:
            result.append(s)
    return " ".join(result)

def generate_structural_noise(paragraph_text, seed=None):
    """Применяет все три техники: HTML-мусор, OCR, обрывы."""
    rng = random.Random(seed)
    result = paragraph_text
    result = inject_html_junk(result, n_injections=rng.randint(1, 3), rng=rng)
    result = apply_ocr_noise(result, error_rate=0.02, rng=rng)
    result = truncate_sentences(result, rng=rng, prob=0.12)
    return result

# Тест
struct_noise = generate_structural_noise(first_gold_text, seed=SEED)
print("Оригинал:")
print(first_gold_text[:250])
print("\nСтруктурный шум:")
print(struct_noise[:400])

Оригинал:
The Port Arthur massacre of 28–29 April 1996 was a massacre in which 35 people were killed and 23 wounded.  It occurred mainly at the historic Port Arthur former prison colony, a popular tourist site in south-eastern Tasmania, Australia.  It was the 

Структурный шум:
The Port Arthur massacre of 28–29 April 1996 was a massacre in which 35 people were killed and 23 wounded. <div class="nav-bar">Home | About | Contact | Login</div> <!-- Ad slot 4712 --> It occurrcd mainly at the historic Port Arthur former prison c0lony, a popular tourist site in south-eastern Tasmania, Australia. Cookie Policy: We use cookies to improve your experience. [Accept] [Reject] It was 


## Шаг 7. Массовая генерация шума для всех 100 вопросов

**Стратегия продолжения:** если ноутбук прервётся на полпути (rate limit, разрыв соединения), при повторном запуске он:
- Загружает существующий `noise_cache.json`, если он есть.
- Пропускает уже обработанные вопросы.
- Продолжает с первого необработанного.

Это критично для работы на free tier с дневным лимитом 500k tokens.

In [18]:
# Загружаем существующий кэш если есть
if NOISE_CACHE_PATH.exists():
    with open(NOISE_CACHE_PATH, encoding="utf-8") as f:
        noise_cache = json.load(f)
    print(f"Загружен существующий noise_cache: {len(noise_cache)} вопросов уже обработаны")
else:
    noise_cache = {}
    print("Кэш пуст, начинаем с нуля")

def save_cache():
    with open(NOISE_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(noise_cache, f, ensure_ascii=False, indent=2)

def process_question(q):
    """Генерирует все 4 типа шума для одного вопроса."""
    qid = q["id"]
    gold = gold_mapping[qid]
    gold_titles = gold["gold_titles"]
    gold_texts_map = gold["gold_texts"]

    entry = {
        "semantic_distractors": [],
        "counterfactuals": [],
        "duplicates": [],
        "structural": [],
    }

    # 1. Семантические дистракторы (без LLM)
    entry["semantic_distractors"] = get_semantic_distractors(qid, n=4)

    # 2. Контрафакты — по одному на каждый gold-параграф
    for gold_title in gold_titles:
        gold_text = gold_texts_map[gold_title]
        cf = generate_validated_counterfactual(q["question"], q["answer"], gold_text)
        if cf is not None:
            entry["counterfactuals"].append({
                "based_on_title": gold_title,
                "original_value": cf.get("original_value"),
                "new_value": cf.get("new_value"),
                "reasoning": cf.get("reasoning"),
                "text": cf.get("new_text"),
                "validation_passed": cf.get("validation_passed", False),
                "attempts": cf.get("attempts"),
                "history": cf.get("history", []),
            })

    # 3. Дубликаты — по одному на каждый gold-параграф
    for gold_title in gold_titles:
        gold_text = gold_texts_map[gold_title]
        dup = generate_validated_duplicate(gold_text)
        if dup is not None:
            entry["duplicates"].append({
                "based_on_title": gold_title,
                "text": dup.get("new_text"),
                "validation_passed": dup.get("validation_passed", False),
                "attempts": dup.get("attempts"),
                "history": dup.get("history", []),
            })

    # 4. Структурный шум — на одном случайном gold-параграфе
    rng_struct = random.Random(SEED + hash(qid) % 10000)
    chosen_title = rng_struct.choice(gold_titles)
    struct_text = generate_structural_noise(gold_texts_map[chosen_title], seed=SEED + hash(qid) % 10000)
    entry["structural"].append({
        "based_on_title": chosen_title,
        "text": struct_text,
    })

    return entry

# Основной цикл с checkpoint'ами
pending = [q for q in questions if q["id"] not in noise_cache]
print(f"К обработке: {len(pending)} из {len(questions)}")

CHECKPOINT_EVERY = 10  # сохраняем каждые 10 вопросов

for i, q in enumerate(tqdm(pending, desc="Генерация шума")):
    try:
        noise_cache[q["id"]] = process_question(q)
        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_cache()
            print(f"  [checkpoint] сохранено {len(noise_cache)} вопросов")
    except Exception as e:
        print(f"  ОШИБКА на {q['id']}: {e}")
        save_cache()
        raise

save_cache()
print(f"\nГотово. В кэше {len(noise_cache)} вопросов")

Загружен существующий noise_cache: 93 вопросов уже обработаны
К обработке: 7 из 100


Генерация шума:   0%|          | 0/7 [00:00<?, ?it/s]

  [rate-limiter] used=4560, est=1317, жду 50.5с...
  [rate-limiter] used=4688, est=1057, жду 50.5с...

Готово. В кэше 100 вопросов


In [19]:
print(f"В noise_cache сейчас: {len(noise_cache)} вопросов")
print(f"Файл на Drive: {NOISE_CACHE_PATH}")
print(f"Существует: {NOISE_CACHE_PATH.exists()}")
if NOISE_CACHE_PATH.exists():
    import json
    with open(NOISE_CACHE_PATH) as f:
        saved = json.load(f)
    print(f"В файле на Drive: {len(saved)} вопросов")

В noise_cache сейчас: 100 вопросов
Файл на Drive: /content/drive/MyDrive/rag_experiment/artifacts/noise_cache.json
Существует: True
В файле на Drive: 100 вопросов


## Шаг 8. Отчёт по валидации

Смотрим, какой процент сгенерированного шума прошёл валидацию. Если < 80% — надо разбираться, что не так.

In [20]:
report = {
    "counterfactuals": {"total": 0, "valid": 0, "invalid": 0, "attempts_distribution": Counter()},
    "duplicates": {"total": 0, "valid": 0, "invalid": 0, "attempts_distribution": Counter()},
    "structural": {"total": 0},
    "semantic_distractors": {"total": 0},
}

invalid_reasons = defaultdict(Counter)

for qid, entry in noise_cache.items():
    for cf in entry["counterfactuals"]:
        report["counterfactuals"]["total"] += 1
        if cf["validation_passed"]:
            report["counterfactuals"]["valid"] += 1
        else:
            report["counterfactuals"]["invalid"] += 1
            # последняя причина из history
            if cf.get("history"):
                last = cf["history"][-1]
                invalid_reasons["counterfactuals"][last.get("reason", "unknown")] += 1
        report["counterfactuals"]["attempts_distribution"][cf.get("attempts", 0)] += 1

    for dup in entry["duplicates"]:
        report["duplicates"]["total"] += 1
        if dup["validation_passed"]:
            report["duplicates"]["valid"] += 1
        else:
            report["duplicates"]["invalid"] += 1
            if dup.get("history"):
                last = dup["history"][-1]
                invalid_reasons["duplicates"][last.get("reason", "unknown")] += 1
        report["duplicates"]["attempts_distribution"][dup.get("attempts", 0)] += 1

    report["structural"]["total"] += len(entry["structural"])
    report["semantic_distractors"]["total"] += len(entry["semantic_distractors"])

# Конвертация Counter'ов для JSON
for key in ["counterfactuals", "duplicates"]:
    report[key]["attempts_distribution"] = dict(report[key]["attempts_distribution"])

report["invalid_reasons"] = {k: dict(v) for k, v in invalid_reasons.items()}

print("=" * 60)
print("ОТЧЁТ ПО ВАЛИДАЦИИ ШУМА")
print("=" * 60)
for noise_type in ["counterfactuals", "duplicates"]:
    r = report[noise_type]
    pct_valid = 100 * r["valid"] / max(r["total"], 1)
    print(f"\n{noise_type}:")
    print(f"  всего: {r['total']}")
    print(f"  валидных: {r['valid']} ({pct_valid:.1f}%)")
    print(f"  невалидных: {r['invalid']}")
    print(f"  распределение по попыткам: {r['attempts_distribution']}")
    if invalid_reasons[noise_type]:
        print(f"  причины отбраковки: {dict(invalid_reasons[noise_type])}")

print(f"\nstructural: всего {report['structural']['total']} (без валидации)")
print(f"semantic_distractors: всего {report['semantic_distractors']['total']} (без валидации, из HotpotQA)")

with open(NOISE_VALIDATION_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"\nОтчёт сохранён: {NOISE_VALIDATION_REPORT_PATH}")

ОТЧЁТ ПО ВАЛИДАЦИИ ШУМА

counterfactuals:
  всего: 200
  валидных: 197 (98.5%)
  невалидных: 3
  распределение по попыткам: {1: 197, 3: 3}
  причины отбраковки: {'new_value not in new_text': 1, 'low cosine similarity (0.726)': 1, 'low cosine similarity (0.717)': 1}

duplicates:
  всего: 200
  валидных: 200 (100.0%)
  невалидных: 0
  распределение по попыткам: {1: 200}

structural: всего 100 (без валидации)
semantic_distractors: всего 393 (без валидации, из HotpotQA)

Отчёт сохранён: /content/drive/MyDrive/rag_experiment/artifacts/noise_validation_report.json


## Шаг 9. Создание коллекции `rag_noisy` для end-to-end режима

Две подзадачи:

1. **Копируем** все 7997 точек из `rag_clean` в новую коллекцию `rag_noisy`.
2. **Инъецируем** шумные документы: контрафакты, дубликаты, структурные варианты. Семантические дистракторы уже есть в корпусе (они приходят из HotpotQA вместе с gold'ами), их не надо дополнительно добавлять.

Каждая инъекция проходит тот же chunking + embedding pipeline, что и исходный корпус.

In [21]:
# Пересоздаём rag_noisy с нуля
existing_collections = [c.name for c in qdrant.get_collections().collections]
if COLLECTION_NOISY in existing_collections:
    print(f"Удаляю существующую коллекцию {COLLECTION_NOISY}")
    qdrant.delete_collection(collection_name=COLLECTION_NOISY)

qdrant.create_collection(
    collection_name=COLLECTION_NOISY,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)
qdrant.create_payload_index(collection_name=COLLECTION_NOISY, field_name="title", field_schema=PayloadSchemaType.KEYWORD)
qdrant.create_payload_index(collection_name=COLLECTION_NOISY, field_name="role", field_schema=PayloadSchemaType.KEYWORD)
print(f"Коллекция {COLLECTION_NOISY} создана")

Коллекция rag_noisy создана


In [22]:
# Копирование через scroll
print("Копирование rag_clean -> rag_noisy...")
offset = None
copied = 0
BATCH = 256
while True:
    points, next_offset = qdrant.scroll(
        collection_name=COLLECTION_CLEAN,
        limit=BATCH,
        offset=offset,
        with_payload=True,
        with_vectors=True,
    )
    if not points:
        break
    to_upsert = [PointStruct(id=p.id, vector=p.vector, payload=p.payload) for p in points]
    qdrant.upsert(collection_name=COLLECTION_NOISY, points=to_upsert, wait=True)
    copied += len(points)
    if next_offset is None:
        break
    offset = next_offset

noisy_info = qdrant.get_collection(COLLECTION_NOISY)
print(f"Скопировано: {copied}, в коллекции сейчас: {noisy_info.points_count}")

Копирование rag_clean -> rag_noisy...
Скопировано: 7997, в коллекции сейчас: 7997


In [24]:
!pip install -q langchain-text-splitters
# Инъекция шумных документов
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

# Собираем все инъекции в один список
injections = []  # dict: text, title, role, source_question_id

for qid, entry in noise_cache.items():
    # Контрафакты: только те, что прошли валидацию
    for i, cf in enumerate(entry["counterfactuals"]):
        if not cf.get("validation_passed"):
            continue
        injections.append({
            "text": cf["text"],
            "title": f"{cf['based_on_title']} [CF-{i}]",
            "role": "injected_counterfactual",
            "source_question_id": qid,
            "based_on_title": cf["based_on_title"],
        })
    # Дубликаты
    for i, dup in enumerate(entry["duplicates"]):
        if not dup.get("validation_passed"):
            continue
        injections.append({
            "text": dup["text"],
            "title": f"{dup['based_on_title']} [DUP-{i}]",
            "role": "injected_duplicate",
            "source_question_id": qid,
            "based_on_title": dup["based_on_title"],
        })
    # Структурные
    for i, st in enumerate(entry["structural"]):
        injections.append({
            "text": st["text"],
            "title": f"{st['based_on_title']} [STR-{i}]",
            "role": "injected_structural",
            "source_question_id": qid,
            "based_on_title": st["based_on_title"],
        })

print(f"Всего инъекций: {len(injections)}")
print(f"  контрафактов: {sum(1 for x in injections if x['role'] == 'injected_counterfactual')}")
print(f"  дубликатов: {sum(1 for x in injections if x['role'] == 'injected_duplicate')}")
print(f"  структурных: {sum(1 for x in injections if x['role'] == 'injected_structural')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 11.7 MB/s eta 0:00:00
Всего инъекций: 497
  контрафактов: 197
  дубликатов: 200
  структурных: 100


In [25]:
# Чанкинг и эмбеддинги для инъекций
injection_chunks = []
for inj in injections:
    parts = splitter.split_text(inj["text"])
    for ci, part in enumerate(parts):
        chunk_id = hashlib.md5(f"{inj['title']}__{ci}__{inj['source_question_id']}".encode()).hexdigest()
        injection_chunks.append({
            "chunk_id": chunk_id,
            "title": inj["title"],
            "chunk_index": ci,
            "text": part,
            "role": inj["role"],
            "source_question_ids": [inj["source_question_id"]],
            "based_on_title": inj["based_on_title"],
        })

print(f"Чанков из инъекций: {len(injection_chunks)}")

# Эмбеддинги
texts = [c["text"] for c in injection_chunks]
vectors = []
for v in tqdm(embedder.embed(texts, batch_size=64), total=len(texts), desc="Embeddings"):
    vectors.append(v.tolist())

# Upload — используем id-шники, не пересекающиеся с существующими
# rag_clean использовал id 0..7996. Начинаем с 100000 для инъекций.
INJECTION_ID_OFFSET = 100000
points = []
for idx, (chunk, vec) in enumerate(zip(injection_chunks, vectors)):
    points.append(PointStruct(
        id=INJECTION_ID_OFFSET + idx,
        vector=vec,
        payload={
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "chunk_index": chunk["chunk_index"],
            "text": chunk["text"],
            "role": chunk["role"],
            "source_question_ids": chunk["source_question_ids"],
            "based_on_title": chunk["based_on_title"],
        }
    ))

for i in tqdm(range(0, len(points), 256), desc="Upload"):
    batch = points[i:i+256]
    qdrant.upsert(collection_name=COLLECTION_NOISY, points=batch, wait=True)

noisy_info = qdrant.get_collection(COLLECTION_NOISY)
print(f"\nrag_noisy теперь: {noisy_info.points_count} точек")

Чанков из инъекций: 678


Embeddings:   0%|          | 0/678 [00:00<?, ?it/s]

Upload:   0%|          | 0/3 [00:00<?, ?it/s]


rag_noisy теперь: 8675 точек


## Шаг 10. Sanity-check end-to-end

Проверяем, что retriever **реально находит** инъецированные документы при обычном запросе. Если контрафакты не попадают в top-10 — end-to-end режим бессмысленный, они просто лежат в базе мёртвым грузом.

In [26]:
def retrieve_noisy(query, top_k=10):
    qvec = list(embedder.embed([query]))[0].tolist()
    hits = qdrant.query_points(
        collection_name=COLLECTION_NOISY,
        query=qvec,
        limit=top_k,
        with_payload=True,
    ).points
    return hits

rng_sc = random.Random(SEED + 7777)
sample = rng_sc.sample(questions, 10)

role_hit_stats = Counter()
gold_recall_stats = []
injection_hit_details = []

for q in sample:
    hits = retrieve_noisy(q["question"], top_k=10)
    roles_in_top = [h.payload.get("role", "unknown") for h in hits]
    titles_in_top = [h.payload["title"] for h in hits]
    for r in roles_in_top:
        role_hit_stats[r] += 1

    # Был ли инъецированный документ, связанный с ЭТИМ вопросом
    own_injections_in_top = [
        h for h in hits
        if h.payload.get("role", "").startswith("injected_")
        and q["id"] in h.payload.get("source_question_ids", [])
    ]

    gold_titles = gold_mapping[q["id"]]["gold_titles"]
    gold_found = sum(1 for g in gold_titles if any(g in t for t in titles_in_top))
    gold_recall_stats.append(gold_found / len(gold_titles))

    injection_hit_details.append({
        "qid": q["id"],
        "question": q["question"][:80],
        "gold_recall_at_10": gold_found / len(gold_titles),
        "own_injections_in_top10": len(own_injections_in_top),
        "injection_roles": [h.payload["role"] for h in own_injections_in_top],
    })

print("Распределение ролей в top-10 на 10 случайных вопросах:")
for role, cnt in role_hit_stats.most_common():
    print(f"  {role}: {cnt}")

print(f"\nСредний Gold Recall@10: {sum(gold_recall_stats)/len(gold_recall_stats):.2f}")
print(f"(ожидаем примерно как в ноутбуке 01)")

print("\nВопросы, где наши инъекции попали в top-10:")
for d in injection_hit_details:
    if d["own_injections_in_top10"] > 0:
        print(f"  Q: {d['question']}")
        print(f"     инъекций в top-10: {d['own_injections_in_top10']}, роли: {d['injection_roles']}")

Распределение ролей в top-10 на 10 случайных вопросах:
  gold: 22
  injected_counterfactual: 22
  injected_duplicate: 20
  distractor: 20
  injected_structural: 9
  background: 7

Средний Gold Recall@10: 1.00
(ожидаем примерно как в ноутбуке 01)

Вопросы, где наши инъекции попали в top-10:
  Q: in 1999, Spirit Halloween LLC was purchased by a mall retailer that has how many
     инъекций в top-10: 5, роли: ['injected_counterfactual', 'injected_duplicate', 'injected_counterfactual', 'injected_duplicate', 'injected_structural']
  Q: Was the Metropolitan Life Insurance Company Tower [Met Life Tower] or the 15 Hud
     инъекций в top-10: 5, роли: ['injected_duplicate', 'injected_counterfactual', 'injected_counterfactual', 'injected_duplicate', 'injected_structural']
  Q: Who is older, Bertalan Farkas or Hans Schlegel?
     инъекций в top-10: 5, роли: ['injected_counterfactual', 'injected_duplicate', 'injected_counterfactual', 'injected_structural', 'injected_duplicate']
  Q: Who founded th

### Интерпретация sanity-check

**Хороший результат:**
- Gold Recall@10 ≈ такой же, как в ноутбуке 01 (0.85+). Инъекции не должны вытеснять gold полностью.
- На 30–70% вопросов хотя бы одна инъекция попадает в top-10. Это показывает, что retriever «видит» наши инъекции как релевантные.

**Плохой результат:**
- Gold Recall@10 упал сильно → инъекции агрессивно вытесняют gold. Это значит, что эксперимент будет необъективным: шум не «смешивается» с сигналом, а заменяет его.
- Инъекции не попадают в top-10 почти нигде → retriever игнорирует их, end-to-end режим покажет «всё хорошо», потому что шума просто не будет в контексте.

В любом из плохих случаев придётся вручную настроить силу инъекций (количество, позиционирование, «зарядку» текстом вопроса).

## Шаг 11. Обновление config.json

Добавляем информацию о созданных артефактах шума.

In [27]:
base_config["noise"] = {
    "types": ["semantic_distractors", "counterfactuals", "duplicates", "structural"],
    "levels": [0, 40, 80],  # проценты шумных документов в контексте при k=5
    "generator_model": GROQ_MODEL,
    "generator_temperature": GROQ_TEMPERATURE,
    "n_counterfactuals_per_gold": N_COUNTERFACTUALS_PER_GOLD,
    "n_duplicates_per_gold": N_DUPLICATES_PER_GOLD,
    "max_generation_attempts": MAX_GENERATION_ATTEMPTS,
    "collection_noisy": COLLECTION_NOISY,
    "noisy_collection_size": noisy_info.points_count,
    "total_injections": len(injection_chunks),
}

with open(ARTIFACTS_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(base_config, f, ensure_ascii=False, indent=2)

print("config.json обновлён")

config.json обновлён


## Шаг 12. Итоговая сводка

In [29]:
print("=" * 60)
print("НОУТБУК 02 ЗАВЕРШЁН")
print("=" * 60)
print(f"\nQdrant коллекции:")
print(f"  rag_clean: {clean_info.points_count} точек (не тронута)")
print(f"  rag_noisy: {noisy_info.points_count} точек (копия + {len(injection_chunks)} инъекций)")

print(f"\nШум сгенерирован для {len(noise_cache)} вопросов:")
print(f"  semantic_distractors (из HotpotQA): {report['semantic_distractors']['total']}")
print(f"  counterfactuals: {report['counterfactuals']['valid']}/{report['counterfactuals']['total']} валидных")
print(f"  duplicates: {report['duplicates']['valid']}/{report['duplicates']['total']} валидных")
print(f"  structural: {report['structural']['total']}")

print(f"\nАртефакты на Drive:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.1f} KB")



НОУТБУК 02 ЗАВЕРШЁН

Qdrant коллекции:
  rag_clean: 7997 точек (не тронута)
  rag_noisy: 8675 точек (копия + 678 инъекций)

Шум сгенерирован для 100 вопросов:
  semantic_distractors (из HotpotQA): 393
  counterfactuals: 197/200 валидных
  duplicates: 200/200 валидных
  structural: 100

Артефакты на Drive:
  config.json: 0.8 KB
  corpus_meta.json: 0.4 KB
  distractor_mapping.json: 467.9 KB
  gold_mapping.json: 100.1 KB
  noise_cache.json: 652.5 KB
  noise_validation_report.json: 0.5 KB
  questions.json: 29.9 KB
